# 🐍 Part 2: Database Enrichment & Data Wrangling via Python
## Refactoring and Enriching the `Listings` Table (`02_update_listings.ipynb`)


## 🎯 Context & Objectives

The goal of this stage is to perform strategic data enrichment on the `listings` table within the `insight_places` database using Python (`02_update_listings.ipynb`). By standardizing property classification to English, establishing a base daily rate (`base_daily_rate`), enforcing temporal creation rules prior to bookings (`created_at`), and structuring quantitative capacity features (`max_guests`, `bedrooms`, `bathrooms`), I will turn this table into a robust property dimension for advanced analytical queries and BI dashboards. 

This notebook represents the second module of a sequential ETL execution, preserving complete relational integrity with the `addresses` and `bookings` tables.


## 🛠️ Environment Setup & Dependencies

To execute this ETL pipeline securely and efficiently, the following Python libraries are required: 
* **`sqlalchemy` & `pymysql`**: Database abstraction layer and MySQL driver for executing DDL/DML operations. 
* **`python-dotenv`**: Security protocol to parse database credentials from the existing `.env` environment file without exposing sensitive keys. 
* **`pandas`**: Data manipulation, aggregation, and structural validation. 
* **`random` & `numpy`**: Native and mathematical modules used to generate realistic pricing distributions and conditional business logic. 


## 📌 Refining the `Listings` Table 

This module focuses on upgrading property attributes, pricing models, and structural integrity: 

1. **Language Standardization & Category Expansion (`property_type`):** Translating existing types to English (`House`, `Apartment`, `Hotel`) and expanding to include high-end types (`Villa`, `Studio` and `Penthouse`). 
2. **Base Daily Rate Calibration (`base_daily_rate`):** Setting the fixed base rate according to property capacity and spatial attributes (`location_type` from `addresses`), incorporating location premium pricing for categories like `Beach` and `Near Airport`. 
3. **Temporal Integrity Enforcer (`created_at`):** Generating property creation timestamps that strictly precede the first booking start date (`start_date` from `bookings`). 
4. **Structured Capacity Attributes:** Separating property features into dedicated numeric columns (`max_guests`, `bedrooms`, `bathrooms`) for precise filtering in BI. 
5. **Operational Status (`is_active`):** Calibrating the availability boolean flag (`is_active`) to a ~85% active ratio to reflect real-world platform distribution.
6. **Textual Content Enrichment (`description`):** Creating concise, realistic property descriptions to complete the dimension metadata.



___________________________________________________________________________________________________________________________________________________________________________________________
### 1) Installing and importing the libraries

In this step, I will load the libraries that will be used throughout the project.

In [1]:
!pip install sqlalchemy
!pip install pymysql

import pandas as pd
import random
import datetime
from sqlalchemy import create_engine, text
import pymysql


#### 1.1) Installing the data protection library
I will install the `python-dotenv` library to ensure the security of the MySQL password.

In [2]:
!pip install python-dotenv

#### 1.2) Environment Setup and Security Validation

In [3]:
# Installing and Importing libraries
import os
from dotenv import load_dotenv

# Loading variables from the .env file
load_dotenv()

# Assigning credentials to variables using os.getenv
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST', 'localhost')
port = os.getenv('DB_PORT', '3306')
database = os.getenv('DB_NAME', 'insight_places')

# Security Check: Validating if environment variables are present
if all([user, password, host, database]):
    print("🔒 Environment variables loaded successfully.")
    print(" System ready for database connection.")
else:
    print("❌ Error: One or more credentials are missing from the .env file.")

🔒 Environment variables loaded successfully.
 System ready for database connection.


#### 1.3) Database Engine Initialization
Using the validated credentials, I will use SQLAlchemy to create the engine. This object will be the communication bridge between Python and MySQL, allowing for efficient data transfer.

In [4]:
# Creating the database connection engine using validated credentials
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")
print("🚀 Engine connection is ready!")

🚀 Engine connection is ready!


## 2) Execution: Refining and Enriching the `Listings` Table

## 2.1 Schema Adjustment

Adding new attributes to the `listings` table to support capacity breakdown, base pricing, temporal mapping, and text metadata.

In [5]:
def ensure_listings_columns():
    alter_queries = [
        "ALTER TABLE listings ADD COLUMN base_daily_rate DECIMAL(10, 2) NULL;",
        "ALTER TABLE listings ADD COLUMN created_at DATETIME NULL;",
        "ALTER TABLE listings ADD COLUMN bedrooms INT NULL;",
        "ALTER TABLE listings ADD COLUMN beds INT NULL;",
        "ALTER TABLE listings ADD COLUMN bathrooms INT NULL;",
        "ALTER TABLE listings ADD COLUMN max_guests INT NULL;",
        "ALTER TABLE listings ADD COLUMN description VARCHAR(255) NULL;"
    ]
    
    with engine.begin() as conn:
        for query in alter_queries:
            try:
                conn.execute(text(query))
            except Exception:
                # Column already exists
                pass
    print("✅ Schema update verified: columns are ready in 'listings' table.")

ensure_listings_columns()

✅ Schema update verified: columns are ready in 'listings' table.


## 2.2 Property Types & Physical Structure

Standardizing property categories into English (`Apartment`, `House`, `Hotel`, `Studio`, `Villa`, `Penthouse`) and assigning realistic, proportional capacity metrics (`bedrooms`, `beds`, `bathrooms`, `max_guests`).

In [6]:
def update_property_types_and_structure():
    with engine.begin() as conn:
        print("Fetching listings to update structural attributes...")
        listings = conn.execute(text("SELECT listing_id FROM listings")).fetchall()
        
        property_types = ['Apartment', 'House', 'Hotel', 'Studio', 'Villa', 'Penthouse']
        property_weights = [0.45, 0.30, 0.08, 0.10, 0.04, 0.03]
        
        updates = []
        
        for (listing_id,) in listings:
            p_type = random.choices(property_types, weights=property_weights)[0]
            
            # Rule logic based on property type
            if p_type == 'Studio':
                bedrooms = 1
                beds = random.choice([1, 2])
                bathrooms = 1
                max_guests = random.choice([1, 2])
            elif p_type == 'Hotel':
                bedrooms = 1
                beds = random.choice([1, 2])
                bathrooms = 1
                max_guests = random.choice([2, 3])
            elif p_type == 'Apartment':
                bedrooms = random.choice([1, 2, 3])
                beds = bedrooms + random.choice([0, 1])
                bathrooms = random.choice([1, 2])
                max_guests = bedrooms * 2
            elif p_type == 'House':
                bedrooms = random.choice([2, 3, 4])
                beds = bedrooms + random.choice([0, 1, 2])
                bathrooms = random.choice([2, 3])
                max_guests = bedrooms * 2 + 1
            elif p_type == 'Villa':
                bedrooms = random.choice([3, 4, 5])
                beds = bedrooms + random.choice([1, 2, 3])
                bathrooms = random.choice([3, 4, 5])
                max_guests = bedrooms * 2 + 2
            else:  # Penthouse
                bedrooms = random.choice([3, 4])
                beds = bedrooms + random.choice([1, 2])
                bathrooms = random.choice([3, 4])
                max_guests = bedrooms * 2
                
            updates.append({
                "listing_id": listing_id,
                "property_type": p_type,
                "bedrooms": bedrooms,
                "beds": beds,
                "bathrooms": bathrooms,
                "max_guests": max_guests
            })
            
        update_query = text("""
            UPDATE listings 
            SET property_type = :property_type,
                bedrooms = :bedrooms,
                beds = :beds,
                bathrooms = :bathrooms,
                max_guests = :max_guests
            WHERE listing_id = :listing_id
        """)
        
        conn.execute(update_query, updates)
        print("✅ Property types and structural attributes updated successfully!")

update_property_types_and_structure()

Fetching listings to update structural attributes...
✅ Property types and structural attributes updated successfully!


## 2.3 Base Daily Rate Calibration

Calculating the fixed `base_daily_rate` for each property based on its `property_type`, capacity, and spatial location premiums (`location_type` from `addresses`).

In [7]:
def update_base_daily_rates():
    with engine.begin() as conn:
        print("Fetching spatial context from addresses and listings data...")
        
        # Mapping address_id -> location_type
        address_locations = dict(
            conn.execute(text("SELECT address_id, location_type FROM addresses")).fetchall()
        )
        
        # Fetching listings to calculate rates
        listings = conn.execute(
            text("SELECT listing_id, address_id, property_type, bedrooms, max_guests FROM listings")
        ).fetchall()
        
        location_multipliers = {
            'Beach': 1.40,
            'Near Airport': 1.25,
            'City': 1.10,
            'Mountain': 1.15,
            'Countryside': 0.90
        }
        
        updates = []
        
        for listing_id, address_id, p_type, bedrooms, max_guests in listings:
            loc_type = address_locations.get(address_id, 'City')
            
            # Unit rate logic per property type
            if p_type == 'Studio':
                base_price_unit = random.uniform(120, 200)
            elif p_type == 'Hotel':
                base_price_unit = random.uniform(180, 320)
            elif p_type == 'Apartment':
                base_price_unit = 140 + (bedrooms * 70)
            elif p_type == 'House':
                base_price_unit = 200 + (bedrooms * 80)
            elif p_type == 'Villa':
                base_price_unit = 450 + (bedrooms * 130)
            else:  # Penthouse
                base_price_unit = 550 + (bedrooms * 160)
                
            # Applying spatial location premium multiplier
            multiplier = location_multipliers.get(loc_type, 1.0)
            final_rate = round(base_price_unit * multiplier, 2)
            
            updates.append({
                "listing_id": listing_id,
                "base_daily_rate": final_rate
            })
            
        update_query = text("""
            UPDATE listings 
            SET base_daily_rate = :base_daily_rate
            WHERE listing_id = :listing_id
        """)
        
        conn.execute(update_query, updates)
        print("✅ Base daily rates updated successfully with spatial pricing premiums!")

update_base_daily_rates()

Fetching spatial context from addresses and listings data...
✅ Base daily rates updated successfully with spatial pricing premiums!


## 2.4 Temporal Integrity (`created_at`)

Generating creation timestamps (`created_at`) that strictly precede the earliest booking date (`start_date` from `bookings`) for each property, ensuring temporal coherence across relational tables.

In [8]:
def update_created_at_timestamps():
    with engine.begin() as conn:
        print("Fetching minimum booking dates per listing from 'bookings' table...")
        
        # Querying the earliest start_date for each listing
        min_booking_dates = dict(
            conn.execute(
                text("SELECT listing_id, MIN(start_date) FROM bookings GROUP BY listing_id")
            ).fetchall()
        )
        
        listings = conn.execute(text("SELECT listing_id FROM listings")).fetchall()
        
        updates = []
        
        for (listing_id,) in listings:
            min_date = min_booking_dates.get(listing_id)
            
            if min_date:
                if isinstance(min_date, str):
                    min_date = datetime.datetime.strptime(min_date, "%Y-%m-%d").date()
                elif isinstance(min_date, datetime.datetime):
                    min_date = min_date.date()
                
                # Created between 30 and 365 days BEFORE the first booking
                days_prior = random.randint(30, 365)
                created_date = min_date - datetime.timedelta(days=days_prior)
            else:
                # Default fallback date for listings with no booking history
                created_date = datetime.date(2022, 1, 1) + datetime.timedelta(days=random.randint(0, 500))
            
            created_at_str = f"{created_date} 09:00:00"
            
            updates.append({
                "listing_id": listing_id,
                "created_at": created_at_str
            })
            
        update_query = text("""
            UPDATE listings 
            SET created_at = :created_at
            WHERE listing_id = :listing_id
        """)
        
        conn.execute(update_query, updates)
        print("✅ Creation timestamps ('created_at') updated with strict temporal integrity!")

update_created_at_timestamps()

Fetching minimum booking dates per listing from 'bookings' table...
✅ Creation timestamps ('created_at') updated with strict temporal integrity!


## 2.5 Textual Content Enrichment (`description`)

Generating concise, realistic property descriptions tailored to each `property_type` and room count to complete the dimension metadata for listing pages and BI tooltips.

In [9]:
def update_descriptions():
    with engine.begin() as conn:
        print("Fetching property types and bedroom counts to generate descriptions...")
        listings = conn.execute(
            text("SELECT listing_id, property_type, bedrooms FROM listings")
        ).fetchall()
        
        updates = []
        
        for listing_id, p_type, bedrooms in listings:
            if p_type == 'Studio':
                desc = "Cozy and modern studio, perfect for solo travelers or couples."
            elif p_type == 'Hotel':
                desc = "Comfortable hotel suite with full service and prime accessibility."
            elif p_type == 'Apartment':
                desc = f"Spacious {bedrooms}-bedroom apartment with great city views and amenities."
            elif p_type == 'House':
                desc = f"Charming {bedrooms}-bedroom family house with private space."
            elif p_type == 'Villa':
                desc = "Luxury villa featuring high-end amenities, private pool, and scenic views."
            else:  # Penthouse
                desc = "Exclusive penthouse with panoramic skyline views and premium design."
                
            updates.append({
                "listing_id": listing_id,
                "description": desc
            })
            
        update_query = text("""
            UPDATE listings 
            SET description = :description
            WHERE listing_id = :listing_id
        """)
        
        conn.execute(update_query, updates)
        print("✅ Property descriptions generated and updated successfully!")

update_descriptions()

Fetching property types and bedroom counts to generate descriptions...
✅ Property descriptions generated and updated successfully!


## 2.6 Operational Status Inspection (`is_active`)

Executing a diagnostic query to verify the baseline ratio of active (`1`) versus inactive (`0`) properties prior to recalibration.

In [10]:
def check_is_active_distribution():
    query = """
        SELECT 
            is_active,
            COUNT(*) AS total_listings,
            ROUND((COUNT(*) * 100.0 / (SELECT COUNT(*) FROM listings)), 2) AS percentage
        FROM listings
        GROUP BY is_active;
    """
    
    df_distribution = pd.read_sql(query, engine)
    
    print("--- Current 'is_active' Distribution Baseline ---")
    print(df_distribution.to_string(index=False))

check_is_active_distribution()

--- Current 'is_active' Distribution Baseline ---
 is_active  total_listings  percentage
         0            4919       49.27
         1            5065       50.73


## 2.7 Operational Status Recalibration (`is_active`)

The baseline diagnosis revealed an unrealistic ~50/50 split between active and inactive properties. To better reflect real-world short-term rental platforms—where the vast majority of catalog listings remain operational—we recalibrate the `is_active` boolean flag to achieve an approximate 85% active ratio.

In [11]:
def recalibrate_is_active():
    with engine.begin() as conn:
        print("Fetching listings to recalibrate active status...")
        listings = conn.execute(text("SELECT listing_id FROM listings")).fetchall()
        
        updates = []
        
        for (listing_id,) in listings:
            # Assigning 1 with 85% probability, 0 with 15% probability
            is_active_val = 1 if random.random() < 0.85 else 0
            
            updates.append({
                "listing_id": listing_id,
                "is_active": is_active_val
            })
            
        update_query = text("""
            UPDATE listings 
            SET is_active = :is_active
            WHERE listing_id = :listing_id
        """)
        
        conn.execute(update_query, updates)
        print("✅ 'is_active' flag recalibrated to ~85% active ratio!")

recalibrate_is_active()

# Verification query
check_is_active_distribution()

Fetching listings to recalibrate active status...
✅ 'is_active' flag recalibrated to ~85% active ratio!
--- Current 'is_active' Distribution Baseline ---
 is_active  total_listings  percentage
         1            8495       85.09
         0            1489       14.91
